In [1]:
import os
from langgraph.graph import StateGraph, START, END  ## stategraph create the agentic flow , reterive evalute and generate
from langchain_groq import ChatGroq 
from typing import TypedDict, List
from langchain_community.vectorstores import FAISS
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.schema import Document
from langchain_huggingface import HuggingFaceEmbeddings


C:\Users\Sir\AppData\Local\Temp\ipykernel_22684\3702872387.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [2]:
import os 
from langchain_groq import ChatGroq
from dotenv import load_dotenv
load_dotenv()
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2")  ## This model converts your text into numerical vectors that represent its meaning. 

#### llm ini
llm=ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0
)

c:\Users\Sir\OneDrive\Desktop\rag 3\.venv-1\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5450.31it/s]


In [3]:
sample_text= """ Elon Musk is a technology entrepreneur and engineer known for founding SpaceX and Tesla.
He was born on June 28, 1971, in Pretoria, South Africa.
His major achievements include advancing space exploration and electric vehicles.
Musk is also involved with Neuralink and The Boring Company.
This document provides a brief overview of Musk's background and accomplishments.
What Is Retrieval-Augmented Generation (RAG)?
RAG is a hybrid architecture that augments a large language model’s (LLM) text generation capabilities by retrieving and integrating relevant external information from documents, databases or knowledge bases.

Instead of solely relying on the LLM’s internal parameters, the model queries an external retriever.
The retriever searches through a large corpus, returning document snippets semantically relevant to the user's query.
The LLM conditions its generation on both the input query and these retrieved documents.
This process grounds the LLM’s output in concrete, up-to-date data, leading to more factual and contextually correct answers.
Benefits of RAG
Improves accuracy and reduces hallucinations by relying on real data.
Enables LLMs with smaller context windows to access large knowledge bases.
Supports easy updates of knowledge without retraining large models.
Ideal for domain-specific assistants, question-answering, chatbots with factual grounding.
Major Components of a RAG System
RAG-architecture
RAG Architecture
Document Loader and Chunker : Raw documents (text, PDFs, web pages) are loaded and split into smaller, manageable chunks for indexing.
Embeddings Model : Converts text chunks and user queries into semantic vector representations capturing meaning.
Vector Store (Index) : Stores embeddings and supports efficient similarity search to retrieve relevant chunks.
Retriever : Queries the vector store to find top relevant chunks based on the user’s input.
Generative LLM : Takes the query plus retrieved chunks as input and generates a grounded, coherent response.
Chain/Orchestration : Integrates all above components into an end-to-end pipeline, managing retrieval, prompt construction and generation.
Memory (optional but important) : Maintains conversation history or other contextual information for multi-turn interactions (covered in the LangChain-specific implementation).
LangChain: A Modular Framework for RAG
LangChain is a Python SDK designed to build LLM-powered applications offering easy composition of document loading, embedding, retrieval, memory and large model invocation. LangChain’s modular architecture makes assembling RAG pipelines straightforward.

"""

documents=Document(page_content=sample_text,
                    metadata={"source" :"article"})


 ### chunks time 
text_spillter=RecursiveCharacterTextSplitter(
  chunk_size = 500,
  chunk_overlap =50 ) ### in how many part chunks will be done , it just create in spliter , not split
sample_text_split = text_spillter.split_documents([documents])
print(len(sample_text_split))


### reterival time baby make vectorstore
vectorstore = FAISS.from_documents(
    documents=sample_text_split,
    embedding=embeddings
)
retriever = vectorstore.as_retriever(
    search_kwargs={"k":3}
)




### got an error sapmle text split not defined so now split it into chunks 

## time to reterive 

7


In [4]:
def route_question(state):
    question = state["question"]
    context = state.get("context", "")

    prompt = f"""
    Decide whether the question needs information from the document database.

    If it needs information from the documents, return only: retrieve
    If it can be answered without the documents, return only: direct

    Question: {question}
    """

    response = llm.invoke(prompt)
    decision = response.content.strip().lower()

    if decision == "retrieve":
        return "retrieve"
    elif decision == "direct":
        return "direct"
    else:
        return "retrieve"

def retrieve_documents(state):
    documents = retriever.invoke(state["question"])

    context = "\n\n".join(
        doc.page_content for doc in documents
    )

    return {"context": context}

def generate_answer(state):
    question = state["question"]
    context = state.get("context", "")

    prompt = f"""Answer the question using the context below if provided.
Context:
{context}

Question: {question}
Answer:"""
    response = llm.invoke(prompt)
    return {"answer": response.content}


Now create the gragh so that your app (workflow ) gets define 


In [5]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

class AgentState(TypedDict, total=False):
    question: str
    context: str
    answer: str

# Create the graph builder
builder = StateGraph(AgentState)

# Add nodes
builder.add_node("retrieve", retrieve_documents)
builder.add_node("generate", generate_answer)

# Define routing and flow
builder.add_conditional_edges(
    START,
    route_question,
    {
        "retrieve": "retrieve",
        "direct": "generate"
    }
)
builder.add_edge("retrieve", "generate")
builder.add_edge("generate", END)

# Compile the graph
app = builder.compile()


In [11]:
### now ask the question
question = input("please ask the question dying to answer : ")

result = app.invoke({
    "question": question
})

print(result["answer"])


Elon Musk is a billionaire entrepreneur and business magnate best known for founding and leading several high‑profile technology companies. He co‑founded PayPal, founded SpaceX (a private aerospace manufacturer and space transport services company), and is the CEO and lead designer of Tesla, Inc., which produces electric vehicles and renewable energy products. Musk has also been involved with ventures such as Neuralink (neurotechnology), The Boring Company (infrastructure and tunnel construction), and X.com (an early online payment company that became PayPal). He is widely recognized for his ambitious vision of reducing global carbon emissions, enabling human colonization of Mars, and advancing artificial intelligence and transportation technologies.
